In [5]:
import pandas as pd

In [8]:
df = pd.read_csv(r'C:\Users\13429\OneDrive\桌面\archive\events.csv')

In [9]:
df.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [10]:
df.info

<bound method DataFrame.info of              timestamp  visitorid event  itemid  transactionid
0        1433221332117     257597  view  355908            NaN
1        1433224214164     992329  view  248676            NaN
2        1433221999827     111016  view  318965            NaN
3        1433221955914     483717  view  253185            NaN
4        1433221337106     951259  view  367447            NaN
...                ...        ...   ...     ...            ...
2756096  1438398785939     591435  view  261427            NaN
2756097  1438399813142     762376  view  115946            NaN
2756098  1438397820527    1251746  view   78144            NaN
2756099  1438398530703    1184451  view  283392            NaN
2756100  1438400163914     199536  view  152913            NaN

[2756101 rows x 5 columns]>

In [11]:
df.describe()

,timestamp,visitorid,itemid,transactionid
count,2.756101e+06,2.756101e+06,2.756101e+06,22457.000000
mean,1.436424e+12,7.019229e+05,2.349225e+05,8826.497796
std,3.366312e+09,4.056875e+05,1.341954e+05,5098.996290
min,1.430622e+12,0.000000e+00,3.000000e+00,0.000000
25%,1.433478e+12,3.505660e+05,1.181200e+05,4411.000000
50%,1.436453e+12,7.020600e+05,2.360670e+05,8813.000000
75%,1.439225e+12,1.053437e+06,3.507150e+05,13224.000000
max,1.442545e+12,1.407579e+06,4.668670e+05,17671.000000


In [12]:
df['event'].value_counts()

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [16]:
df['timestamp'] = pd.to_datetime(df['timestamp'],unit='ms')
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour


In [17]:
df.isnull().sum()

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
date                   0
hour                   0
dtype: int64

In [18]:
df = df.drop_duplicates()

In [20]:
df['event'].value_counts()

event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: int64

In [21]:
df['visitorid'].nunique()

1407580

In [22]:
df['itemid'].nunique()

235061

In [23]:
dau = df.groupby('date')['visitorid'].nunique()

In [24]:
dau.head()

date
2015-05-03     7865
2015-05-04    11019
2015-05-05    13084
2015-05-06    13687
2015-05-07    13086
Name: visitorid, dtype: int64

In [25]:
view=2664218
cart=68966
buy=22457

In [26]:
view_to_cart = cart/view
cart_to_buy = buy/cart
view_to_buy = buy/view

In [27]:
view_to_cart,cart_to_buy,view_to_buy

(0.025886019837715983, 0.3256242206304556, 0.00842911503488078)

In [28]:
df_buy = df[df['event'] == 'transaction']

In [29]:
rfm = df_buy.groupby('visitorid').agg({
    'timestamp':'max',
    'visitorid':'count'
})

In [30]:
rfm.columns = ['last_time','frequency']
rfm.head

<bound method NDFrame.head of                         last_time  frequency
visitorid                                   
172       2015-08-15 01:29:01.686          2
186       2015-08-12 16:34:57.040          1
264       2015-09-07 17:34:45.708          2
419       2015-07-29 05:03:12.695          1
539       2015-06-16 05:39:38.673          1
...                           ...        ...
1406787   2015-06-01 16:56:48.042          1
1406981   2015-05-18 22:28:21.266          1
1407070   2015-05-13 22:41:15.160          1
1407110   2015-08-04 23:11:25.624          1
1407398   2015-07-07 03:47:15.969          1

[11719 rows x 2 columns]>

In [31]:
import datetime

In [33]:
current_time = df['timestamp'].max()
rfm['recency'] = (current_time - rfm['last_time']).dt.days

In [36]:
rfm['R_score'] = pd.qcut(rfm['recency'],4,labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'),4,labels=[1,2,3,4])

In [37]:
rfm

,last_time,frequency,recency,R_score,F_score
visitorid,,,,,
172,2015-08-15 01:29:01.686,2,34,4,4
186,2015-08-12 16:34:57.040,1,36,4,1
264,2015-09-07 17:34:45.708,2,10,4,4
419,2015-07-29 05:03:12.695,1,50,3,1
539,2015-06-16 05:39:38.673,1,93,2,1
...,...,...,...,...,...
1406787,2015-06-01 16:56:48.042,1,108,1,4
1406981,2015-05-18 22:28:21.266,1,122,1,4
1407070,2015-05-13 22:41:15.160,1,127,1,4


In [39]:
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)

In [44]:
def user_level(row):
    if row['R_score']>=3 and row['F_score']>=3:
        return '高价值用户'
    elif row['R_score']>=3 and row['F_score']<3:
        return '潜力用户'
    elif row['R_score']<3 and row['F_score']>=3:
        return '重要保持用户'
    else:
        return'流失用户'

In [45]:
rfm['user_type'] = rfm.apply(user_level,axis=1)

In [46]:
rfm['user_type'].value_counts()

user_type
高价值用户     2981
流失用户      2963
潜力用户      2897
重要保持用户    2878
Name: count, dtype: int64

In [47]:
rfm

,last_time,frequency,recency,R_score,F_score,user_type
visitorid,,,,,,
172,2015-08-15 01:29:01.686,2,34,4,4,高价值用户
186,2015-08-12 16:34:57.040,1,36,4,1,潜力用户
264,2015-09-07 17:34:45.708,2,10,4,4,高价值用户
419,2015-07-29 05:03:12.695,1,50,3,1,潜力用户
539,2015-06-16 05:39:38.673,1,93,2,1,流失用户
...,...,...,...,...,...,...
1406787,2015-06-01 16:56:48.042,1,108,1,4,重要保持用户
1406981,2015-05-18 22:28:21.266,1,122,1,4,重要保持用户
1407070,2015-05-13 22:41:15.160,1,127,1,4,重要保持用户


In [53]:
from sklearn.preprocessing import StandardScaler
X = rfm[['recency','frequency']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [54]:
from sklearn.cluster import KMeans
KMeans = KMeans(n_clusters=4,random_state=42)
rfm['cluster'] = KMeans.fit_predict(X_scaled)

In [55]:
rfm.groupby('cluster')[['recency','frequency']].mean()

,recency,frequency
cluster,,
0,35.327334,1.593612
1,102.489850,1.469052
2,2.000000,398.000000
3,65.461538,87.256410


In [56]:
rfm['cluster'].value_counts()

cluster
1    6010
0    5667
3      39
2       3
Name: count, dtype: int64